# Investigating Data Leakage in Loan Default Prediction

### Lending Club Loan Dataset, 2007–2018

This project investigates how data leakage can artificially inflate the performance of a machine learning model for loan default prediction.

Three feature conditions are compared:

1. Clean model: uses information available at loan origination.
2. Temporal leakage model: adds information generated after loan origination.
3. Strong leakage model: adds multiple post-origination variables.

The objective is to measure how much model performance changes when information that would not have been available at prediction time is introduced.

## 1. Load the Dataset

This cell imports the libraries required for data handling and machine learning, then loads the Lending Club dataset from the compressed CSV file.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve

dataset_path = "data/accepted_2007_to_2018Q4.csv.gz"

df = pd.read_csv(
    dataset_path,
    compression="gzip",
    low_memory=False
)

print("Dataset shape:", df.shape)

Dataset shape: (2260701, 151)


## 2. Dataset Inspection

Before defining the prediction target and feature sets, the dataset is inspected to understand its basic structure, loan outcomes, and time-related information.

In [ ]:
print("\nAll dataset columns:")

for i, column in enumerate(df.columns, start=1):
    print(f"{i:3}. {column}")

print("\nLoan status counts:")
print(df["loan_status"].value_counts(dropna=False))

print("\nIssue date column:")
print("Data type:", df["issue_d"].dtype)
print("Missing values:", df["issue_d"].isna().sum())

print("\nFirst 10 issue dates:")
print(df["issue_d"].head(10).tolist())

print("\nUnique value examples:")
print(df["issue_d"].dropna().unique()[:10])


All dataset columns:
  1. id
  2. member_id
  3. loan_amnt
  4. funded_amnt
  5. funded_amnt_inv
  6. term
  7. int_rate
  8. installment
  9. grade
 10. sub_grade
 11. emp_title
 12. emp_length
 13. home_ownership
 14. annual_inc
 15. verification_status
 16. issue_d
 17. loan_status
 18. pymnt_plan
 19. url
 20. desc
 21. purpose
 22. title
 23. zip_code
 24. addr_state
 25. dti
 26. delinq_2yrs
 27. earliest_cr_line
 28. fico_range_low
 29. fico_range_high
 30. inq_last_6mths
 31. mths_since_last_delinq
 32. mths_since_last_record
 33. open_acc
 34. pub_rec
 35. revol_bal
 36. revol_util
 37. total_acc
 38. initial_list_status
 39. out_prncp
 40. out_prncp_inv
 41. total_pymnt
 42. total_pymnt_inv
 43. total_rec_prncp
 44. total_rec_int
 45. total_rec_late_fee
 46. recoveries
 47. collection_recovery_fee
 48. last_pymnt_d
 49. last_pymnt_amnt
 50. next_pymnt_d
 51. last_credit_pull_d
 52. last_fico_range_high
 53. last_fico_range_low
 54. collections_12_mths_ex_med
 55. mths_since_

## 3. Define the Prediction Target and Feature Sets

The analysis focuses on two final loan outcomes: **Fully Paid** and **Charged Off**.

These are converted into a binary target where 0 represents a fully paid loan and 1 represents a charged-off loan.

Three feature sets are then created to compare normal prediction with different levels of data leakage:

- **Clean:** features intended to represent information available around loan origination.
- **Temporal leakage:** the clean features plus `last_fico_range_low`.
- **Strong leakage:** the temporal set plus `total_pymnt`, `recoveries`, and `last_pymnt_amnt`.

## 4. Prepare the Data for Modeling

The dataset is divided into training and testing sets using the same split for all three experiments.

The preprocessing pipeline handles missing values, scales numerical variables, and converts categorical variables into numerical form. Keeping the same split and preprocessing approach allows the three models to be compared fairly.

## 5. Train the Models

Three logistic regression models are trained using the same training observations:

- Clean: uses only information available at loan origination.
- Temporal leakage: adds a variable containing information generated later in the loan lifecycle.
- Strong leakage: adds several variables that reflect subsequent loan performance.

This allows the effect of data leakage on model performance to be measured directly.

## 6. Evaluate Model Performance

Each model is evaluated on the same unseen test set using ROC-AUC and accuracy.

ROC-AUC measures how well the model distinguishes between fully paid and charged-off loans, while accuracy measures the proportion of test observations classified correctly.

## 7. Compare Model Performance

The following visualization compares ROC-AUC across the three feature conditions. A large increase in performance after introducing post-origination information provides evidence of the effect of data leakage.

## 8. Examine the Leakage Features

The leakage variables are compared across the two loan outcomes using their median values.

These variables contain information generated during the later stages of a loan, so large differences between the two outcome groups help explain why including them can substantially improve apparent model performance.

## 9. ROC Curve Comparison

The ROC curves provide a visual comparison of model discrimination under the three feature conditions.

The comparison shows how introducing information generated after loan origination changes the apparent predictive performance.

## 10. Conclusion

The clean model achieved a ROC-AUC of 0.696, providing the baseline performance using information available at loan origination.

Introducing the temporal leakage feature increased ROC-AUC to 0.945, an increase of 0.249, while the strong leakage model achieved a ROC-AUC of 0.998, an increase of 0.302 compared with the clean model.

These results demonstrate that using information generated after the prediction point can substantially inflate the apparent performance of a machine learning model. Variables such as `recoveries`, `total_pymnt`, and `last_pymnt_amnt` are particularly problematic because they describe events occurring during the later lifecycle of the loan.

Therefore, when developing a model for loan default prediction, features must be restricted to information that would actually be available at the intended prediction time. This investigation shows how data leakage can make a model appear highly accurate while relying on information that would not be available when making the real-world prediction.